# RoboFactory expert

> RoboFactory's motion-planning expert as a policy, to collect expert datasets.

In [ ]:
#| default_exp envs.robofactory.expert

In [ ]:
#| hide
from nbdev.showdoc import *

RoboFactory solves each task with a scripted motion planner (grasp poses from annotated objects,
screw motions and RRT plans computed with [mplib](https://github.com/haosulab/MPlib)), which is how its
and MangoBench's datasets are made. `RoboFactoryExpert` makes it a stable-marl policy: at the first step of
an episode it plans on a private copy of the env, put in the same simulation state, records the
planner's per-step `pd_joint_pos` actions and then plays them; after the plan (or when planning failed,
`plan_failed`) the arms hold their joints. The CPU simulation is deterministic, so the plan
replayed in the env does what it did in the copy.

**numpy < 2.** ManiSkill pins mplib 0.1.1, built against numpy 1: with numpy 2 it crashes the
interpreter, so the expert refuses to plan there. Collect expert data in an environment with
`numpy<2` (e.g. `pip install 'stable-marl[robofactory]' 'numpy<2'`, without the `gcrl` extra, whose
JAX needs numpy 2); the environments themselves work with either.

In [ ]:
#| export
from __future__ import annotations

import numpy as np

from stable_marl.policy import ExpertPolicy

In [ ]:
#| export
class _Recorder:
    """
    The planner's view of the private env: `reset` lands on the target state (whatever the seed),
    `step` records the actions (one row per arm).
    """
    def __init__(self, env, state, agent_names):
        self.env, self.state, self.agent_names, self.actions = env, state, agent_names, []

    @property
    def unwrapped(self):
        return self.env

    def reset(self, seed=None, options=None):
        out = self.env.reset(seed=seed)
        self.env.set_state(self.state)
        return out

    def step(self, action):
        if isinstance(action, dict):
            self.actions.append(np.stack([np.asarray(action[n], np.float32).reshape(-1) for n in self.agent_names]))
        else:
            self.actions.append(np.asarray(action, np.float32).reshape(1, -1))
        return self.env.step(action)

    def __getattr__(self, name):
        return getattr(self.env, name)


class RoboFactoryExpert(ExpertPolicy):
    """
    RoboFactory's motion-planning solution of the task, as a policy of a `World` of
    `RoboFactoryEnv` (see above; needs numpy < 2). `plan_failed[i]` / `plan_success[i]`: whether
    planning failed / whether the plan solved the task in the private copy, for env `i`'s current episode.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._plans: dict[int, dict] = {}
        self._planning_envs: dict[tuple, object] = {}

    def _planning_env(self, env):
        "A private copy of `env`'s task without cameras, shared by the envs of the same task."
        key = (env.task, env.scene)
        if key not in self._planning_envs:
            import gymnasium as gym
            self._planning_envs[key] = gym.make(f'{env.task}-rf', config=env.config, obs_mode='none',
                                                control_mode='pd_joint_pos', reward_mode='sparse',
                                                sim_backend='cpu', render_mode=None).unwrapped
        return self._planning_envs[key]

    def plan(self, env) -> dict:
        """
        Plan the current episode of `env` (a `RoboFactoryEnv`): `actions` (T, num_agents, 8), `failed`, `success`.
        """
        if np.lib.NumpyVersion(np.__version__) >= '2.0.0':
            raise RuntimeError("RoboFactory's motion planner (mplib 0.1.1) crashes with numpy 2: "
                               "collect expert data in an environment with numpy<2")
        import contextlib, io, torch
        from stable_marl.envs.robofactory.upstream.planner.run import MP_SOLUTIONS
        base = self._planning_env(env)
        rec = _Recorder(base, torch.as_tensor(env.state())[None], env.agent_names)
        with contextlib.redirect_stdout(io.StringIO()):
            try:
                res = MP_SOLUTIONS[f'{env.task}-rf'](rec, seed=env._seed)
            except Exception:                 # planning errors (no IK solution, collisions) count as failures
                res = -1
        failed = isinstance(res, int) and res == -1
        success = not failed and bool(np.asarray(res[-1]['success']).reshape(-1)[0])
        actions = np.stack(rec.actions) if rec.actions else np.zeros((0, env.num_agents, 8), np.float32)
        return {'actions': actions, 'failed': failed, 'success': success}

    def act(self, env) -> np.ndarray:
        plan = self._plans.get(id(env))
        if plan is None or plan['episode'] != env._episode:
            plan = self._plans[id(env)] = {'episode': env._episode, 't': 0, **self.plan(env)}
        t = plan['t']
        plan['t'] += 1
        if t < len(plan['actions']):
            return plan['actions'][t]
        hold = env.hold_action()
        if len(plan['actions']):
            hold[:, -1] = plan['actions'][-1, :, -1]         # keep the last gripper command (e.g. holding an object)
        return hold

    @property
    def plan_failed(self) -> list[bool]:
        return [self._plans.get(id(e.unwrapped), {}).get('failed', False) for e in self.env.envs]

    @property
    def plan_success(self) -> list[bool]:
        return [self._plans.get(id(e.unwrapped), {}).get('success', False) for e in self.env.envs]

In [ ]:
show_doc(RoboFactoryExpert.plan)

### Example

In [ ]:
#| eval: false
# needs numpy < 2 (mplib 0.1.1): run in the data-collection environment
from stable_marl import World
world = World('RoboFactory-LiftBarrier-v0', num_envs=1, camera_size=(64, 64))
expert = RoboFactoryExpert()
world.set_policy(expert)
res = world.evaluate(episodes=2, seed=0)
print(res['success_rate'], res['episode_lengths'], expert.plan_success)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()